In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()

spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
orders_df = spark.read.parquet("data/external_table/part-00000-6109de51-a0e8-41af-839d-58e5ef769898-c000.snappy.parquet", inferSchema=True, header=True)

In [3]:
orders_df.createOrReplaceTempView('orders')

### 2. Total count of customers with 'CLOSED' order status

In [4]:
spark.sql("SELECT count(distinct(customer_id)) AS customer_count FROM orders where order_status='CLOSED'")

customer_count
5655


In [5]:
from pyspark.sql.functions import countDistinct
orders_df.filter("order_status = 'CLOSED'").countDistinct('customer_id')

AttributeError: 'DataFrame' object has no attribute 'countDistinct'

In [6]:
count = orders_df.filter("order_status = 'CLOSED'").select('customer_id').distinct().count()
# Note: count() returns a single value: integerType

In [7]:
print(count)

5655


In [8]:
orders_df.filter("order_status = 'CLOSED'").select(countDistinct('customer_id'))

count(DISTINCT customer_id)
5655
